# Emotion Detection with YOLOv12

A simple, end-to-end project that trains a **YOLOv12** object detector to recognise facial emotions.

**Classes:** `anger`, `fear`, `happy`, `neutral`, `sad`, `surprise`

The notebook covers:
1. Setup & dataset configuration
2. Training YOLOv12 on the emotion dataset
3. Validating the trained model
4. Two reusable prediction helpers: **`predict_single`** (one image) and **`predict_batch`** (many images)
5. Sample predictions drawn from the `valid/` folder

## 1. Setup

Install [Ultralytics](https://docs.ultralytics.com/) (which ships YOLOv12). Uncomment the line below the first time you run this notebook.

In [11]:
# %pip install ultralytics

In [12]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import ultralytics
from ultralytics import YOLO

ultralytics.checks()

# Project paths
PROJECT_DIR = Path.cwd()
DATASET_DIR = PROJECT_DIR / "dataset"
VALID_IMAGES_DIR = DATASET_DIR / "valid" / "images"
PRED_OUTPUT_DIR = PROJECT_DIR / "predictions"
PRED_OUTPUT_DIR.mkdir(exist_ok=True)

print("Project dir :", PROJECT_DIR)
print("Dataset dir :", DATASET_DIR)

Ultralytics 8.4.90  Python-3.11.9 torch-2.8.0+cpu CPU (12th Gen Intel Core(TM) i5-1235U)
Setup complete  (12 CPUs, 15.7 GB RAM, 88.6/219.7 GB disk)
Project dir : d:\codes\repos\Emotix
Dataset dir : d:\codes\repos\Emotix\dataset


## 2. Dataset configuration

The dataset ships with a `data.yaml`, but its image paths are relative in a way that breaks when training from this folder. We write a small corrected config with an **absolute** `path` so training works reliably.

In [13]:
CLASS_NAMES = ["anger", "fear", "happy", "neutral", "sad", "surprise"]

data_yaml = DATASET_DIR / "data_fixed.yaml"
data_yaml.write_text(
    f"path: {DATASET_DIR.as_posix()}\n"
    "train: train/images\n"
    "val: valid/images\n"
    "test: test/images\n"
    f"nc: {len(CLASS_NAMES)}\n"
    f"names: {CLASS_NAMES}\n"
)
print(data_yaml.read_text())

path: d:/codes/repos/Emotix/dataset
train: train/images
val: valid/images
test: test/images
nc: 6
names: ['anger', 'fear', 'happy', 'neutral', 'sad', 'surprise']



## 3. Train YOLOv12

We start from the pretrained `yolo12n.pt` (nano) checkpoint and fine-tune it on the emotion dataset. Settings are kept modest so it trains in a reasonable time on CPU; increase `EPOCHS` / `IMGSZ` (or use a GPU) for better accuracy.

In [14]:
EPOCHS = 100
IMGSZ = 320
BATCH = 8

model = YOLO("yolo12n.pt")  # pretrained YOLOv12 nano

# Print the epoch number to the console at the end of every training epoch
def print_epoch(trainer):
    print(f"==> Completed epoch {trainer.epoch + 1}/{trainer.epochs}", flush=True)

model.add_callback("on_train_epoch_end", print_epoch)

results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(PROJECT_DIR / "runs"),  # absolute -> outputs land in runs/emotion_yolov12
    name="emotion_yolov12",
    exist_ok=True,
)

best_weights = Path(model.trainer.best)
print("Best weights saved to:", best_weights)

New https://pypi.org/project/ultralytics/8.4.92 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.90  Python-3.11.9 torch-2.8.0+cpu CPU (12th Gen Intel Core(TM) i5-1235U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\codes\repos\Emotix\dataset\data_fixed.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, mu

## 4. Validate the trained model

Reload the best checkpoint and evaluate it on the validation split.

In [15]:
model = YOLO(str(best_weights))
metrics = model.val(data=str(data_yaml), imgsz=IMGSZ)

print(f"mAP50    : {metrics.box.map50:.3f}")
print(f"mAP50-95 : {metrics.box.map:.3f}")

Ultralytics 8.4.90  Python-3.11.9 torch-2.8.0+cpu CPU (12th Gen Intel Core(TM) i5-1235U)
YOLOv12n summary (fused): 159 layers, 2,557,898 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 168.444.3 MB/s, size: 36.9 KB)
val: Scanning D:\codes\repos\Emotix\dataset\valid\labels.cache... 155 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 155/155  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.0s/it 10.4s1ss
                   all        155        824      0.482       0.63      0.566      0.421
                 anger         88        195      0.628       0.61      0.706       0.53
                  fear         39        106      0.531      0.642      0.558       0.44
                 happy         94        255      0.744      0.833      0.851      0.613
               neutral         31         87      0.181      0.517      0.238      0.171
                   sad         71   

## 5. Prediction helpers

Two small functions:

- **`predict_single`** &mdash; run detection on one image and (optionally) display / save it.
- **`predict_batch`** &mdash; run detection on a list of images and show them in a grid.

Both return the raw Ultralytics `Results` so you can access boxes, classes and confidences programmatically.

In [16]:
def predict_single(model, image_path, conf=0.25, show=True, save_path=None):
    """Detect emotions in a single image.

    Args:
        model:      a loaded YOLO model.
        image_path: path to one image.
        conf:       confidence threshold.
        show:       if True, display the annotated image inline.
        save_path:  if given, write the annotated image to this path.

    Returns:
        The Ultralytics Results object for the image.
    """
    result = model.predict(source=str(image_path), conf=conf, verbose=False)[0]
    annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)  # BGR -> RGB

    if save_path is not None:
        cv2.imwrite(str(save_path), cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))

    if show:
        plt.figure(figsize=(5, 5))
        plt.imshow(annotated)
        plt.title(Path(image_path).name)
        plt.axis("off")
        plt.show()

    # Print a short text summary of what was detected
    for box in result.boxes:
        cls = model.names[int(box.cls)]
        print(f"  {cls}: {float(box.conf):.2f}")

    return result

In [17]:
def predict_batch(model, image_paths, conf=0.25, cols=3, save_dir=None):
    """Detect emotions in a list of images and show them in a grid.

    Args:
        model:       a loaded YOLO model.
        image_paths: list of image paths.
        conf:        confidence threshold.
        cols:        number of columns in the display grid.
        save_dir:    if given, save each annotated image into this folder.

    Returns:
        The list of Ultralytics Results objects.
    """
    image_paths = [str(p) for p in image_paths]
    results = model.predict(source=image_paths, conf=conf, verbose=False)

    rows = (len(results) + cols - 1) // cols
    plt.figure(figsize=(cols * 4, rows * 4))

    for i, result in enumerate(results):
        annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)

        if save_dir is not None:
            out = Path(save_dir) / f"pred_{Path(result.path).name}"
            cv2.imwrite(str(out), cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))

        plt.subplot(rows, cols, i + 1)
        plt.imshow(annotated)
        plt.title(Path(result.path).name, fontsize=8)
        plt.axis("off")

    plt.tight_layout()
    plt.show()
    return results

## 6. Sample predictions from the validation set

Grab a handful of images from `valid/images` and run both helpers on them. Annotated copies are written to the `predictions/` folder.

In [18]:
sample_images = sorted(VALID_IMAGES_DIR.glob("*.jpg"))[:6]
print("Sample images:")
for p in sample_images:
    print(" -", p.name)

# Batch prediction on the samples
batch_results = predict_batch(model, sample_images, conf=0.25, cols=3, save_dir=PRED_OUTPUT_DIR)

Sample images:
 - 14_jpg.rf.0dc3ebf389b2649f7fb02ac4e3473f68.jpg
 - 16_jpg.rf.4ab6be837aed23fa8bbe5099d6088f3f.jpg
 - 17_jpg.rf.afbee266294337a94745e3412331e756.jpg
 - 19_jpg.rf.0d5ebf951f3fc659a16b6ac82a8d4e0c.jpg
 - 1_jpg.rf.d098c11a4f287e5b95d3fb8900b7b485.jpg
 - 20_jpg.rf.bd6d7050470677823d0be2ef28f28b4e.jpg


<Figure size 1200x800 with 6 Axes>

In [19]:
# Single-image prediction on the first sample
single_result = predict_single(
    model,
    sample_images[0],
    conf=0.25,
    save_path=PRED_OUTPUT_DIR / "single_prediction.jpg",
)

<Figure size 500x500 with 1 Axes>

  happy: 0.86
  happy: 0.80
  sad: 0.79
  anger: 0.75
  fear: 0.55
  surprise: 0.54
  anger: 0.49
  happy: 0.49
  fear: 0.48
  fear: 0.40
  neutral: 0.36
  neutral: 0.34
  anger: 0.34
  neutral: 0.30
  fear: 0.26
  sad: 0.26
  neutral: 0.26
  neutral: 0.26


## 7. Export & save models

Export the trained detector to **ONNX** (for portable / cross-framework inference) and gather
every generated model — `best.pt`, `last.pt`, and `best.onnx` — into a single `Models/` folder.

In [20]:
from shutil import copy2

MODELS_DIR = PROJECT_DIR / "Models"
MODELS_DIR.mkdir(exist_ok=True)

# Export the trained model to ONNX (writes best.onnx next to best.pt)
onnx_path = YOLO(str(best_weights)).export(format="onnx", imgsz=IMGSZ)

# Collect every generated model into the Models/ folder
weights_dir = best_weights.parent
copy2(weights_dir / "best.pt", MODELS_DIR / "best.pt")
copy2(weights_dir / "last.pt", MODELS_DIR / "last.pt")
copy2(onnx_path, MODELS_DIR / "best.onnx")

print("Saved models to", MODELS_DIR)
for m in sorted(MODELS_DIR.iterdir()):
    print(f"  {m.name}  ({m.stat().st_size / 1e6:.1f} MB)")

Ultralytics 8.4.90  Python-3.11.9 torch-2.8.0+cpu CPU (12th Gen Intel Core(TM) i5-1235U)
YOLOv12n summary (fused): 159 layers, 2,557,898 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from 'D:\codes\repos\Emotix\runs\emotion_yolov12\weights\best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 10, 2100) (5.2 MB)

ONNX: starting export with onnx 1.22.0 opset 22...


C:\Users\RISHABH\AppData\Roaming\Python\Python311\site-packages\torch\onnx\utils.py:1397: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  warnings.warn(


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success  4.9s, saved as 'D:\codes\repos\Emotix\runs\emotion_yolov12\weights\best.onnx' (10.0 MB)

Export complete (5.6s)
Results saved to D:\codes\repos\Emotix\runs\emotion_yolov12\weights\best.onnx
Predict:         yolo predict task=detect model=D:\codes\repos\Emotix\runs\emotion_yolov12\weights\best.onnx imgsz=320 
Validate:        yolo val task=detect model=D:\codes\repos\Emotix\runs\emotion_yolov12\weights\best.onnx imgsz=320 data=d:\codes\repos\Emotix\dataset\data_fixed.yaml  
Visualize:       https://netron.app
Saved models to d:\codes\repos\Emotix\Models
  best.onnx  (10.4 MB)
  best.pt  (5.5 MB)
  last.pt  (5.5 MB)


---
That's it &mdash; the trained weights live under `runs/emotion_yolov12/weights/best.pt`, and annotated sample predictions are in `predictions/`.